# [재설계 · 2026-10-26] 같은 계층 구조(β 풀 + 라우터)를 **1일** 주기로 재학습 (`1d`)

**배경.** 30분 주기 v4에서 (1) 30분 판단은 노이즈라 스무딩 없이는 비용이 수익을 전부 잠식했고(α=1 비용 후 −24.8%),
(2) 효과가 있던 것은 판단을 사실상 ~2일 평균으로 만든 α=0.01뿐이었으며, (3) 라우터 붕괴 원인은 30분 신호 부족(IC≈0)으로 분석됐다.
EarnHFT의 성과는 수수료 0~0.02% 전제(원 논문 5.3절·부록 표 1)였고, 이 프로젝트는 0.1%다.

**가설.** 구조는 그대로 두고 의사결정 주기만 늘리면 신호 대비 비용·노이즈가 줄어 (a) 정책 로짓의 IC가 0보다 커지고
(b) 라우터가 하나의 전문가로 붕괴하지 않을 것이다. 어느 쪽으로 나와도 30분 결과의 원인 해석을 검증한다.

## 사전 등록한 설계 (실행 전에 고정)
| 항목 | 값 |
|---|---|
| 주기 | **1일(`1d`)로 고정** — 다른 주기는 `crypto_horizon_retrain_4h.ipynb` (동시에 다른 Colab 세션에서 실행 가능) |
| 학습 | 2021-01-01 ~ 2025-09-30 (기존 v4는 2025년만 — 상승장 포함하도록 확장) |
| 검증 | 2025-10-01 ~ 12-31 (**학습에 포함하지 않음** — 기존 v4보다 깨끗한 분할) |
| 테스트 | 2026-01-01 ~ 07-13 (보고용) · 2026-07-14 이후 미지 구간은 **이 노트북에서 쓰지 않는다** |
| 그대로 둔 것 | 임베딩 모델, top-K=10+게이트, β 4개, PPO 하이퍼파라미터·스텝 수, 라우터(DQN, 매칭) 설정, 비용 0.1% |
| 주기에 맞춰 바꾼 것 | 버킷 길이, 수익률 타겟(다음 버킷까지), 라우팅 단위 = 1일, 라우터 관측 창 = 2일(1d는 최소 14버킷), 청크 기울기를 30분 기준으로 환산(β의 의미 유지) |
| 원본 스텝 수를 유지한 것 | 레짐 분할 저역통과·최소 구간 길이 = 48버킷(4h는 8일, 1d는 48일) — 1일로 줄이면 1d에서 구간이 1버킷까지 쪼개져 PPO 에피소드가 사라진다 |
| 선택 규칙 | 전략(β 단독 4개·라우터) × α ∈ {1, 0.3, 0.1, 0.03, 0.01} 중 **검증 샤프 최고**를 고르고 테스트 결과를 그대로 보고 |
| 보조(선택과 무관) | 선택된 전략과 β=30 α=1을 수수료 0% / 0.015% / 0.02%(EarnHFT 원 논문 기준) / 0.1%로 재계산 |
| 평가 | 2주 워밍업(시작 상태 영향 제거), 비교: BTC 보유·등가중·30분 v4(β=30 워밍업: 검증 −0.08% / 테스트 +2.86%) |

## 실행
- **런타임 유형을 GPU(T4)로 바꾸고** `런타임 > 모두 실행`만 누르면 된다(GPU가 없으면 CPU로도 되지만 임베딩 생성이 느리다). 임베딩 생성은 GPU, PPO 학습은 CPU를 쓴다.
- 예상 시간: 임베딩 ~20~40분(4h) / ~10분(1d) + PPO 4개·라우터 3~5시간. 끊기면 다시 `모두 실행` — 임베딩과 학습된 모델은 Drive에서 이어받는다.
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/hz_{HORIZON}/crypto_horizon_eval.csv`, 모델: `models/hz_{HORIZON}/`


In [ ]:
HORIZON = '1d'   # 이 노트북은 1일 주기 전용 — 바꾸지 말 것
BUCKET_OF = {'4h': 4 * 3600, '1d': 24 * 3600}
assert HORIZON in BUCKET_OF
HZ_SEC = BUCKET_OF[HORIZON]
HZ_BARS = HZ_SEC // 300                 # 5분봉 개수 = 수익률 타겟 길이
DAY_BUCKETS = 86400 // HZ_SEC           # 하루의 버킷 수
print(f'주기 {HORIZON}: 버킷 {HZ_SEC}초 = 5분봉 {HZ_BARS}개, 하루 {DAY_BUCKETS}버킷')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow chronos-forecasting einops


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = f'hz_{HORIZON}'   # 주기별 별도 폴더 — 기존 rt1006 모델·결과를 건드리지 않는다
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


## 1. 주기별 임베딩 (GPU)

In [ ]:
%%time
# 1. 주기별 임베딩 생성 — 기존 crypto_hybrid_model_colab 셀 20과 같은 정규화·같은 모델, 버킷 경계(UTC) 봉에서만 추출
import sys, subprocess, glob, h5py, numpy as np, pandas as pd, torch
from pathlib import Path
REPO = Path('/content/ToBeRich')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/winbean-kwon/ToBeRich.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '55bce50d4e3f7049177bdf43d4f7c43f4bbe9772'], check=True)
sys.path.insert(0, str(REPO))
from crypto.preprocess_features import FEATURE_COLS as PF_FEATURE_COLS
from crypto.models.hybrid_model import load_hybrid_model, load_chronos_pipeline, extract_chronos_embs

DRIVE = Path('/content/drive/MyDrive/졸업프로젝트')
FEAT_DIR_ = DRIVE / 'data' / 'crypto' / 'features_5m'
HZ_EMB_H5 = DRIVE / 'data' / 'crypto' / f'crypto_rl_embeddings_{HORIZON}.h5'
EMB_START = pd.Timestamp('2020-12-01', tz='UTC')   # 2021-01-01 학습 시작 전 여유
EMB_END = pd.Timestamp('2026-07-14', tz='UTC')     # 미지 구간(07-14~)은 만들지 않는다
SEQ = 60
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if HZ_EMB_H5.exists():
    print('기존 임베딩 재사용:', HZ_EMB_H5)
else:
    hybrid = load_hybrid_model(DRIVE / 'models' / 'crypto_hybrid_best.pt', device=DEVICE)
    pipe = load_chronos_pipeline(device=DEVICE)
    S, T, E = [], [], []
    for fp in sorted(glob.glob(str(FEAT_DIR_ / '*.parquet'))):
        sym = Path(fp).stem
        df = pd.read_parquet(fp, columns=['datetime'] + PF_FEATURE_COLS)
        ts = df['datetime'].astype('int64').to_numpy() // 10**9
        feat = df[PF_FEATURE_COLS].to_numpy(np.float32)
        close = feat[:, 0]
        cand = np.where((ts % HZ_SEC == 0) & (ts >= EMB_START.timestamp()) & (ts < EMB_END.timestamp()))[0]
        cand = cand[cand >= SEQ - 1]
        for k in range(0, len(cand), 1024):
            b = cand[k:k + 1024]
            win = np.stack([feat[i - SEQ + 1:i + 1] for i in b])
            Xb = (win - win.mean(1, keepdims=True)) / (win.std(1, keepdims=True) + 1e-8)
            Cb = np.stack([close[i - SEQ + 1:i + 1] for i in b])
            c_emb = torch.tensor(extract_chronos_embs(pipe, Cb, batch_size=1024, device=DEVICE), device=DEVICE)
            with torch.no_grad():
                e = hybrid.embed(torch.tensor(Xb, device=DEVICE), c_emb).cpu().numpy()
            S += [sym] * len(b); T += ts[b].tolist(); E.append(e)
        print(f'{sym}: {len(cand):,}개')
        del df, feat
    E = np.concatenate(E).astype(np.float32)
    with h5py.File(HZ_EMB_H5, 'w') as f:
        f.create_dataset('symbols', data=np.array(S, dtype='S12'), compression='gzip')
        f.create_dataset('timestamps', data=np.array(T, dtype=np.int64), compression='gzip')
        f.create_dataset('embeddings', data=E, compression='gzip', chunks=(1000, 64))
    print(f'저장: {HZ_EMB_H5} ({len(E):,}개)')
    del hybrid, pipe
    torch.cuda.empty_cache()


## 2. β 풀 + 매칭 라우터 학습 (원본 코드 재사용)

In [ ]:
# 2. 원본 v4 블록 실행 (커밋 0e86a41의 코드 그대로, 위 표의 값만 치환) — 데이터·환경·레짐 분할 → β 풀 4개 → 매칭 라우터
run_cells('crypto/notebooks/crypto_hrl_earnhft_colab.ipynb', [2, 4, 6, 8, 9, 11, 12], patch_after=2, subs={2: [('BUCKET_SEC = 1800     # 30분 버킷', 'BUCKET_SEC = HZ_SEC   # [재설계] 주기별 버킷'), ("RL_TRAIN_START = '2025-01-01'", "RL_TRAIN_START = '2021-01-01'"), ("RL_TRAIN_END   = '2025-12-31'", "RL_TRAIN_END   = '2025-09-30'   # [재설계] 검증(10~12월)을 학습에서 제외"), ("RL_EMB_H5      = CRYPTO_DIR / 'crypto_rl_embeddings.h5'", "RL_EMB_H5      = CRYPTO_DIR / f'crypto_rl_embeddings_{HORIZON}.h5'"), ('ROUTE_BUCKETS         = 48 ', 'ROUTE_BUCKETS         = DAY_BUCKETS '), ('MACRO_WINDOW           = 96 ', 'MACRO_WINDOW           = max(2 * DAY_BUCKETS, 14) ')], 6: [("df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])", "df = pd.read_parquet(fp, columns=['datetime', 'close'])\n    df['RetTarget_6b'] = df['close'].shift(-HZ_BARS) / df['close'] - 1   # [재설계] 다음 버킷까지의 수익률(5분 격자는 빈틈 없음)")]})
_ip.run_cell("# [재설계] 청크 기울기를 30분 기준으로 환산 — chunk_sampling_priority의 β·기울기 항이 주기와 무관하게 같은 의미를 갖도록\nseg.chunks['slope'] = seg.chunks['slope'] * (1800 / BUCKET_SEC)\nprint(f'청크 {len(seg.chunks)}개, 기울기 30분 기준 환산 완료')\n")
run_cells('crypto/notebooks/crypto_hrl_earnhft_colab.ipynb', [44, 46, 48, 50, 63], patch_after=None)
print('학습 완료')


## 3. 사전 등록한 평가

In [ ]:
%%time
# 3. 평가 — 2주 워밍업, α 그리드, β 단독 4개 + 매칭 라우터, BTC·등가중, IC, 라우터 붕괴 여부
from scipy.stats import spearmanr
STEPS_PER_YEAR = 365 * DAY_BUCKETS
WARMUP = 14 * DAY_BUCKETS
ALPHAS = [1.0, 0.3, 0.1, 0.03, 0.01]
valid_buckets = [b for b in all_buckets if '2025-10-01' <= bucket_to_date(b) <= '2025-12-31']
test_buckets = [b for b in all_buckets if '2026-01-01' <= bucket_to_date(b) <= '2026-07-13']
print(f'검증 {len(valid_buckets)}스텝 · 테스트 {len(test_buckets)}스텝 · 워밍업 {WARMUP}스텝')


def perf(values, turns=None):
    v = np.asarray(values, dtype=np.float64)
    r = np.diff(v) / v[:-1]
    out = {'총 수익률': v[-1] / v[0] - 1, 'Sharpe': r.mean() / (r.std() + 1e-12) * np.sqrt(STEPS_PER_YEAR),
           'MDD': (v / np.maximum.accumulate(v) - 1).min()}
    if turns is not None:
        out['평균 턴오버/스텝'] = float(np.mean(turns))
    return out


def _warm(buckets):
    i0 = all_buckets.index(buckets[0]); lo = max(0, i0 - WARMUP)
    return all_buckets[lo:i0] + list(buckets), i0 - lo


def run_solo(model, alpha, buckets):
    run_bk, skip = _warm(buckets)
    env = SparseGatedCryptoPortfolioEnv(buckets=run_bk, top_k=TOP_K, **env_kwargs)
    env.reset(); env.weights = np.zeros(env.n_assets, dtype=np.float32); obs = env._get_obs()
    expo, done = [], False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        tw, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * tw
        expo.append(float(w.sum()))
        obs, _, done, _, _ = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    v = np.asarray(env.value_history[skip:]); return v / v[0], env.turnover_history[skip:], np.mean(expo[skip:])


def run_router(router, alpha, buckets):
    """원본 run_v4_router_smoothed_on과 같은 루프 + 워밍업. 평가 구간에서 고른 β 분포도 반환."""
    run_bk, skip = _warm(buckets)
    env = MacroRouterEnvV4(buckets=run_bk, pool_models=[pool_models_v4[b] for b in POOL_BETAS],
                           route_buckets=ROUTE_BUCKETS, macro_window=MACRO_WINDOW, top_k=TOP_K,
                           episode_progress_mode='constant', **env_kwargs)
    obs, _ = env.reset(); inner = env.inner
    inner.weights = np.zeros(inner.n_assets, dtype=np.float32)
    chosen, done = [], False
    while not done:
        act, _ = router.predict(obs, deterministic=True)
        if inner.t >= skip:
            chosen.append(POOL_BETAS[int(act)])
        model = env.pool_models[int(act)]; done_inner = False
        for _ in range(env.route_buckets):
            a, _ = model.predict(inner._get_obs(), deterministic=True)
            tw, rho, top_idx = inner.action_to_weights(a)
            w = (1 - alpha) * inner.weights + alpha * tw
            _, _, done_inner, _, info = inner.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
            env._nav_history.append(info['portfolio_value'])
            if done_inner:
                break
        obs = env._macro_obs(); done = done_inner or inner.t >= len(inner.buckets) - 1
    v = np.asarray(inner.value_history[skip:]); return v / v[0], inner.turnover_history[skip:], pd.Series(chosen).value_counts(normalize=True).to_dict()


def equal_weight_backtest(env):
    """원본 셀 21과 동일 — 매 스텝 등가중으로 재조정(거래 불가 종목 제외)."""
    obs, _ = env.reset()
    done, values = False, [1.0]
    action = np.ones(env.n_assets, dtype=np.float32) / env.n_assets
    while not done:
        obs, _, done, _, info = env.step(action)
        values.append(info['portfolio_value'])
    return np.array(values), list(env.turnover_history)


def step_ic(model, buckets):
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **env_kwargs)
    obs, _ = env.reset(); ics, done = [], False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        b = env.buckets[env.t]
        real = np.array([ret_index.get((s, b), np.nan) for s in symbols]); m = ~np.isnan(real)
        if m.sum() >= 5:
            c, _ = spearmanr(np.asarray(a[:-1])[m], real[m])
            if not np.isnan(c):
                ics.append(c)
        tw, rho, top_idx = env.action_to_weights(a)
        obs, _, done, _, _ = env.step_with_weights(tw, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    ics = np.array(ics); return ics.mean(), ics.mean() / (ics.std() / np.sqrt(len(ics)) + 1e-12)


rows = []
for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
    r_btc = np.asarray([ret_index.get(('BTCUSDT', b), 0.0) for b in bk[:-1]])
    rows.append({'구간': split, '전략': 'BTC 보유', 'α': '-', **perf(np.concatenate([[1], np.cumprod(1 + r_btc)])), '평균 노출': 1.0})
    ew_v, ew_t = equal_weight_backtest(CryptoPortfolioEnv(buckets=bk, **env_kwargs))
    rows.append({'구간': split, '전략': '등가중(비용 포함)', 'α': '-', **perf(ew_v, ew_t), '평균 노출': 1.0})
    for alpha in ALPHAS:
        for beta in POOL_BETAS:
            v, t, x = run_solo(pool_models_v4[beta], alpha, bk)
            rows.append({'구간': split, '전략': f'β={beta}', 'α': alpha, **perf(v, t), '평균 노출': x})
        v, t, dist = run_router(router_v4_matched, alpha, bk)
        rows.append({'구간': split, '전략': '라우터(매칭)', 'α': alpha, **perf(v, t), '평균 노출': np.nan,
                     '라우터 최빈 β 비율': max(dist.values()) if dist else np.nan, '라우터 β 분포': str(dist)})
        print(f'{split} α={alpha} 완료')

res = pd.DataFrame(rows)
res.insert(0, '주기', HORIZON)
ic_rows = []
for beta in POOL_BETAS:
    for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
        m, t = step_ic(pool_models_v4[beta], bk)
        ic_rows.append({'주기': HORIZON, '구간': split, 'β': beta, '평균 IC': m, 't': t})
ic_df = pd.DataFrame(ic_rows)
res.to_csv(CRYPTO_DIR / 'crypto_horizon_eval.csv', index=False, encoding='utf-8-sig')
ic_df.to_csv(CRYPTO_DIR / 'crypto_horizon_ic.csv', index=False, encoding='utf-8-sig')

pd.set_option('display.width', 220)
show = res.copy()
for k, f in {'총 수익률': '{:+.2%}', 'Sharpe': '{:.2f}', 'MDD': '{:.1%}', '평균 노출': '{:.1%}'}.items():
    show[k] = show[k].map(lambda x, f=f: f.format(x) if pd.notna(x) else '-')
print(show[['구간', '전략', 'α', '총 수익률', 'Sharpe', 'MDD', '평균 노출', '라우터 최빈 β 비율']].to_string(index=False))
print(); print(ic_df.to_string(index=False, float_format=lambda x: f'{x:+.4f}'))

# 사전 등록한 선택 규칙: 검증 샤프 최고 (전략, α) → 그 테스트 결과를 그대로 보고
cand = res[(res['구간'] == 'valid') & (res['α'] != '-')]
best = cand.loc[cand['Sharpe'].idxmax()]
test_row = res[(res['구간'] == 'test') & (res['전략'] == best['전략']) & (res['α'] == best['α'])].iloc[0]
print(f"\n[선택] 검증 샤프 최고: {best['전략']} α={best['α']} (검증 {best['총 수익률']:+.2%}, 샤프 {best['Sharpe']:.2f})")
print(f"       → 테스트 {test_row['총 수익률']:+.2%}, 샤프 {test_row['Sharpe']:.2f}, MDD {test_row['MDD']:.1%}")
print('       참고: 30분 v4 β=30 워밍업 — 검증 −0.08% / 테스트 +2.86% (샤프 1.03)')
print('저장 완료: crypto_horizon_eval.csv, crypto_horizon_ic.csv')

# 보조(선택 규칙에 영향 없음): EarnHFT 원 논문 수수료 기준 민감도 — 본문 5.3절 0%/0.02%, 부록 표 1 0.015%, 이 프로젝트 0.1%
def with_tc(tc, fn, *args):
    old = env_kwargs['tc']; env_kwargs['tc'] = tc
    try:
        return fn(*args)
    finally:
        env_kwargs['tc'] = old


sens = []
best_alpha = float(best['α'])
for tc in [0.0, 0.00015, 0.0002, 0.001]:
    for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
        if best['전략'] == '라우터(매칭)':
            v, t, _ = with_tc(tc, run_router, router_v4_matched, best_alpha, bk)
        else:
            v, t, _ = with_tc(tc, run_solo, pool_models_v4[int(best['전략'].split('=')[1])], best_alpha, bk)
        sens.append({'주기': HORIZON, '수수료': tc, '구간': split, '전략': f"{best['전략']} α={best_alpha}(선택)", **perf(v, t)})
        v, t, _ = with_tc(tc, run_solo, pool_models_v4[30], 1.0, bk)
        sens.append({'주기': HORIZON, '수수료': tc, '구간': split, '전략': 'β=30 α=1(매 판단 그대로)', **perf(v, t)})
sens = pd.DataFrame(sens)
sens.to_csv(CRYPTO_DIR / 'crypto_horizon_fee_sensitivity.csv', index=False, encoding='utf-8-sig')
print('\n[보조] 수수료 민감도 (EarnHFT 기준 0~0.02% vs 이 프로젝트 0.1%)')
print(sens.to_string(index=False, float_format=lambda x: f'{x:+.4f}'))
